<a href="https://colab.research.google.com/github/GabrielaTranslite/Sentiment_Analysis_Video_Games/blob/main/Reviews_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install steamreviews -q

In [4]:
# ── Step 1: install ────────────────────────────────────────────────────────────
# Run this line separately in a Colab cell first:
# !pip install steamreviews -q

# ── Step 2: mount Google Drive so the CSV survives session restarts ────────────
# Optional but recommended — comment out if you don't want to use Drive.
from google.colab import drive
drive.mount("/content/drive")
OUTFILE = "/content/drive/MyDrive/steam_reviews_3274580.csv"

# If you prefer to skip Drive and just download the file manually at the end,
# replace the two lines above with:
# OUTFILE = "/content/steam_reviews_3274580.csv"

# ── Imports ────────────────────────────────────────────────────────────────────
import steamreviews
import json, csv, os
from datetime import datetime, timezone

APPID      = 3274580
JSON_CACHE = f"/content/jsonfiles/review_{APPID}.json"   # where steamreviews saves data in Colab


# ── Helper utilities ───────────────────────────────────────────────────────────
def unix_to_date(ts):
    return datetime.fromtimestamp(ts, timezone.utc).strftime("%Y-%m-%d")

def hours_from_minutes(minutes):
    try:
        return round(minutes / 60.0, 2)
    except Exception:
        return ""


# ── Download ───────────────────────────────────────────────────────────────────
print("Downloading reviews — this will take a few minutes …")

# No extra params: let steamreviews use its defaults.
# In a FRESH session (no cache) the returned dict contains ALL reviews.
review_dict, query_count = steamreviews.download_reviews_for_app_id(
    APPID,
    chosen_request_params={"language": "english"},
)

print(f"API requests made : {query_count}")

# review_dict is a wrapper with 3 keys: "reviews", "query_summary", "cursors".
# The actual reviews live inside review_dict["reviews"].
reviews = review_dict["reviews"]
print(f"Reviews downloaded: {len(reviews)}")


# ── Write CSV ──────────────────────────────────────────────────────────────────
fieldnames = [
    "review_id", "appid", "author_steamid", "voted_up", "votes_up",
    "votes_funny", "review_text", "timestamp_created", "publish_date",
    "hours_on_record", "weighted_vote_score", "comment_count",
]

with open(OUTFILE, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for rid, r in reviews.items():
        author  = r.get("author", {})
        minutes = (author.get("playtime_at_review")
                   or author.get("playtime_forever")
                   or 0)
        ts = r.get("timestamp_created", 0)

        writer.writerow({
            "review_id":           rid,
            "appid":               APPID,
            "author_steamid":      author.get("steamid", ""),
            "voted_up":            r.get("voted_up", ""),
            "votes_up":            r.get("votes_up", 0),
            "votes_funny":         r.get("votes_funny", 0),
            "review_text":         r.get("review", "").replace("\r", " ").replace("\n", " "),
            "timestamp_created":   ts,
            "publish_date":        unix_to_date(ts) if ts else "",
            "hours_on_record":     hours_from_minutes(minutes),
            "weighted_vote_score": r.get("weighted_vote_score", ""),
            "comment_count":       r.get("comment_count", 0),
        })

print(f"\nDone. Saved {len(reviews)} reviews → {OUTFILE}")

Mounted at /content/drive
[appID = 3274580] expected #reviews = 5269
API requests made : 54
Reviews downloaded: 5268

Done. Saved 5268 reviews → /content/drive/MyDrive/steam_reviews_3274580.csv
